# 第 1 天练习变体 —— 简历摘要器

## 练习目标（理念）

本 notebook 基于 Week 1 Day 1 的「网站摘要」流程，把同一套 **Chat Completions** 模式改用到**简历摘要**场景：

- **输入**：一段简历正文（或先抓取网页文本）
- **输出**：适合 LinkedIn 的一段 bio + 一行 tagline
- **核心 API**：`openai.chat.completions.create(...)`，配合 `messages`（system / user）

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 环境变量 / `.env` | `load_dotenv` + `OPENAI_API_KEY` |
| OpenAI 客户端 | `OpenAI()` → `chat.completions.create` |
| system / user prompt | 先定角色语气，再塞入简历正文 |
| 网页抓取 → 摘要 | `fetch_website_contents` + `summarize` / `display_summary` |
| 商业变体 | 文末把摘要对象从「网站」换成「简历」 |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好项目根目录附近的 `.env`，含 `OPENAI_API_KEY`
3. 确保同目录（或 Python 路径上）有 `scraper.fetch_website_contents`；若缺模块请对照官方 Day 1 / 故障排查
4. 跑通网站摘要后，重点看最后「简历」单元格：可改 `user_prompt` 里的简历文本再试

### 若你刚接触 Notebook

点击代码单元格，按 Shift+Return 执行；请从顶部按顺序运行。指南见课程仓库的 Guides / setup。

### 重要说明

建议在看完讲座**之后**自己完整跑一遍：加 `print`、改 prompt、换 URL / 简历文本，做成你自己的变体。

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">请阅读——重要说明</h2>
            <span style="color:#900;">先建立对 API 调用的直觉，再做自己的变体。社区贡献文件夹就是展示这类变体的好地方。</span>
        </td>
    </tr>
</table>


### 若在 Cursor 中：安装扩展（可选）

1. 菜单 **View → Extensions**
2. 搜索并安装 Microsoft 的 **Python**（`ms-python`）
3. 搜索并安装 **Jupyter**（`ms-toolsai`）

### 选择内核（Kernel）

点击右上角 **Select Kernel** → **Python Environments...** → 选带 `.venv` 的解释器（通常标为 Recommended）。

每个 notebook 都要单独选一次内核。若选不到，去 setup 里的 troubleshooting notebook。


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从本地 scraper 模块导入网页抓取函数：给定 URL，返回清洗后的页面文本
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：在笔记本里用 Markdown 漂亮渲染模型输出
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI

# 若本格报 ImportError / ModuleNotFoundError：先确认内核是课程 .venv，再查 troubleshooting notebook


# 连接到 OpenAI（也可改用 Ollama）

下一格会加载 `.env` 中的环境变量，并检查 `OPENAI_API_KEY` 是否像有效密钥。

若想用免费本地 **Ollama**，请对照 README「付费 API 的免费替代」以及 solutions 里的 `day1_with_ollama.ipynb`。

## 遇错怎么排查

- **NameError**：是否从上到下跑过所有单元格？变量必须先定义再使用
- 仍不行：打开 [troubleshooting](../setup/troubleshooting.ipynb) 逐步诊断
- API 费用：Day 1 调用量很小；也可用 Ollama 作免费替代（Day 2 会细讲）


In [ ]:
# ========== 环境：加载 .env 并做 API Key 健康检查 ==========

# 加载名为 .env 的文件；override=True 表示用文件里的值覆盖已存在的同名环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenAI 密钥（字符串名必须是 OPENAI_API_KEY，与常见 SDK 默认一致）
api_key = os.getenv('OPENAI_API_KEY')

# 下面三段是「键是否存在 / 前缀像不像 / 有没有首尾空白」的教学检查（错误文案保持英文，便于对照官方材料）

if not api_key:
    # 完全没读到密钥
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    # 读到了但不像当前常见的项目密钥前缀（仍可能是旧格式；此处按原逻辑提示）
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    # 首尾有空格/制表符，调用时很容易 401
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    # 初步检查通过（真正能否调用还要看额度与网络）
    print("API key found and looks good so far!")


## 先热身：直接调用一次前沿模型

下面两格用最小 `messages` 结构打通 OpenAI，确认客户端与密钥可用。


In [ ]:
# ========== 预览：构造最简单的 user 消息 ==========

# 发给模型的用户文本（可运行字符串保持英文；改它就会改变模型回复）
message = "Hello, GPT! This is my first ever message to you! Hi!"

# Chat Completions 要求 messages 为「角色 + 内容」字典列表；这里只有一条 user
messages = [{"role": "user", "content": message}]

# 在笔记本里单独写变量名：会显示该对象，方便你目视检查结构
messages


In [ ]:
# ========== 第一次真实 API 调用 ==========

# 创建客户端；无参时默认从环境变量 OPENAI_API_KEY 取密钥
openai = OpenAI()

# chat.completions.create：同步一次生成；model id 保持原样（gpt-5-nano）
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
# choices[0].message.content：取第一条候选回复的文本正文
response.choices[0].message.content


## 开始第一个项目：网站 → 摘要


In [ ]:
# ========== 试用抓取工具 ==========

# 调用 scraper：抓取给定 URL 的可读文本（依赖本机网络与目标站是否允许简单抓取）
ed = fetch_website_contents("https://www.ics.uci.edu")
# 打印抓到的正文，确认后面喂给模型的原料长什么样
print(ed)


## 提示类型（Prompt Types）

像 GPT 这类聊天模型通常期望两类指令：

- **system prompt**：任务是什么、语气如何、输出格式约束
- **user prompt**：本轮具体要处理的内容（网页正文、简历、问题等）

后面会把两者放进同一条 `messages` 列表再调用 API。


In [ ]:
# ========== 系统提示：定角色与输出格式（发给模型的英文勿改译） ==========

# 三引号多行字符串：rude / snarky 语气 + 要求 Markdown、且不要包在代码围栏里
system_prompt = """
You are a rude assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


In [ ]:
# ========== 用户提示前缀：后面会拼接网页正文 ==========

# 前缀说明「要做什么」；真正内容由 messages_for(website) 拼到后面
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


## 消息结构（messages）

OpenAI Chat Completions（以及许多兼容 API）期望：

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```

下面先用一个极简笑话助手例子熟悉格式，再接到真正的网站摘要管线。


In [ ]:
# ========== 最小完整调用：system + user ==========

# 两条消息：system 定「喜剧助手」人格；user 问算术（内容保持英文）
messages = [
    {"role": "system", "content": "You are a comedic assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

# 换一个小模型 id（gpt-4.1-nano）演示同一套 API
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 取出生成文本
response.choices[0].message.content


## 用函数组装「网站摘要」专用 messages


In [ ]:
# ========== messages_for：把 system / user 拼成 API 需要的列表 ==========

# 入参 website：已抓取的页面文本（字符串）
def messages_for(website):
    # 返回值形状与上一格示例相同：先 system，再 user（前缀 + 正文）
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


In [ ]:
# ========== 目视检查：看看拼出来的 messages 长什么样 ==========

# 用前面抓到的 ed 文本试跑函数（只构造，不调用 API）
messages_for(ed)


## 串起来：抓取 → 组 messages → 调 OpenAI


In [ ]:
# ========== summarize：端到端摘要一个 URL ==========

def summarize(url):
    # 1) 抓取网页正文
    website = fetch_website_contents(url)
    # 2) 调用 Chat Completions；model 用 gpt-4.1-mini（字符串保持原样）
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    # 3) 只返回助手文本，方便上层再 display
    return response.choices[0].message.content


In [ ]:
# ========== 试跑：UCI ICS 站点 ==========

summarize("https://www.ics.uci.edu")


In [ ]:
# ========== display_summary：摘要后再用 Markdown 渲染 ==========

def display_summary(url):
    # 先拿到纯文本摘要
    summary = summarize(url)
    # 在笔记本输出区渲染 Markdown（标题/列表会排版）
    display(Markdown(summary))


In [ ]:
# ========== 试跑：CNN ==========

display_summary("https://cnn.com")


## 再试更多网站

注意：这种简单抓取**只对「服务端直接返回 HTML 正文」的站点有效**。

- 大量依赖 JavaScript（如部分 React 应用）时，正文可能抓不全 → 社区贡献里有 Selenium / Playwright 方案
- 被 CloudFront 等防护的站点可能返回 403
- 许多企业官网 / 新闻站仍然可以直接试

下面两格继续调用同一个 `display_summary`。


In [ ]:
# ========== 再跑一次 CNN（可对比语气是否稳定） ==========

display_summary("https://cnn.com")


In [ ]:
# ========== 试跑：Anthropic 官网 ==========

display_summary("https://anthropic.com")


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">你刚第一次调用了 Frontier Model 的云端 API。摘要是经典 GenAI 用例：新闻、财报、求职场景里的<strong>简历 → bio / 标题</strong>都可以套同一模式。想想你业务里哪里需要「长文变短文」。</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">继续之前——亲手改一版</h2>
            <span style="color:#900;">用下一格做你自己的商业小例子。本贡献把摘要对象换成了<strong>简历</strong>：生成 LinkedIn 风格简介 + 一行 tagline。也可改成「读邮件正文 → 建议主题行」。</span>
        </td>
    </tr>
</table>


In [ ]:
# ========== 商业变体：简历 → LinkedIn bio + tagline ==========

# 第 1 步：创建提示（发给模型的英文 system/user 字符串保持原样，改译会改变行为）

# system：规定输出两块——一段 LinkedIn 语气 bio + 一行简历页眉 tagline
system_prompt = "You are a helpful assistant that can analyze the contents of a resume and write: 1.  a one-paragraph bio of the person based on their resume. The bio should be in an appropriate tone to appear in a LinkedIn profile. 2. A one-line tagline for a resume header."
# user：前缀说明任务，后面贴上完整简历正文（示例人物 ROBERT MCTUTTERSON）
user_prompt = """
    Here are the contents of a resume. Create a one-paragraph bio and a one-line tagline:
    ROBERT MCTUTTERSON, PH.D.
| Website | GitHub

•	Recognized strengths in technical writing, teaching, leadership, and program development of modern, industry-relevant course content across the computer science spectrum.
•	Leadership experience includes management of 50+ student software engineering teams; supervision of multiple teaching assistants; and key advisory leadership roles.
•	Software engineer with expertise in Java programming and software design, and experience with a breadth of front- and back-end technologies.
•	Global expert with seminal impact in game-based pedagogy in software engineering education.

PROFESSIONAL EXPERIENCE
University of California, Irvine, Donald Bren School of Information & Computer Sciences	
Continuing Lecturer/Lecturer 	2014–present                          
•	Top-rated instruction: Taught 10,000+ graduate/undergraduate students across 12 courses including Java programming, mobile programming, GUI programming, software design, project management, requirements analysis, data structures/algorithms, how computers work, and critical writing.
o	Earn high teaching evaluations, averaging 3.69/4 for the 2024-25 academic year.
o	Manage teams of ~4-5 Teaching Assistants per course.
o	Lecturer of the Year Honorable Mention (2025, 2024, 2022), Nomination (2019); Excellence in Digital Learning Honorable Mention (2022). 
•	Program innovation: Developed 6 courses and re-designed 8 courses
o	Increased pedagogical effectiveness and inclusiveness with innovative, evidence-based teaching and learning strategies such as active learning.
o	Modernized software engineering project courses to align with industry best practices, such as Scrum, AI, collaboration tools (Github, Trello, others), and professional ethics. 
•	Leadership: Hands-on team management and key advisory roles
o	Managed 50+ high-performance student software engineering teams developing real-world products for external customers in senior capstone project courses.
o	Drive school-wide continuous review and improvement in teaching through serving on the Lecturer Review Board.
o	Serve on program steering committee for Business Information Management degree program for one of UCI’s highest-growth majors.
o	Volunteer judge and faculty advisor for multiple UCI awards programs and student clubs

Mercor Intelligence
Writing Expert (freelance/remote) 	2025–present               
Provided expertise in long form essay writing to improve models for a top AI lab.

Educational Testing SErvice
AP CS Principles Test Scorer (freelance/remote) 	2025 (seasonal)               
Scored AP Computer Science Principles tests.

Chimborazo Publishing                                                    
Instructional Materials Developer – Technical Writer (freelance/remote) 	2020–present               
Develop instructor supplements and assessments for over 600 courses/textbooks on CS/IT topics such as cloud services (AWS, Azure, Google, Alibaba, Red Hat), cybersecurity, programming languages (Java, Python, C#, JavaScript, HTML, CSS, SQL, R), databases, machine learning, networking, Linux, software engineering processes (agile/Scrum, DevOps, version control), data science, data visualization, and others

University of California, Irvine, Donald Bren School of Information & Computer Sciences	
Assistant Project Scientist 	2007–2012                          
•	Researched, designed, and implemented SimSE, an educational software engineering simulation environment (Java application consisting of ~46K lines of code) that generates custom simulation games based on specified software process models
•	Awarded the 2009 Premier Award for Excellence in Education Courseware for SimSE
•	Achieved seminal impact on the field of game-based pedagogy in software engineering education, resulting in SimSE’s use in classrooms globally and 500+ citations in scientific publications.
•	Developed six SimSE process models (waterfall, incremental, inspection, rapid prototyping, Rational Unified Process, Extreme Programming) for execution in the environment
•	Developed a collection of educational resources to accompany SimSE: course modules, instruction manuals, instructional videos, learning objectives, and project suggestions
•	Conducted formative & summative SimSE evaluations w/multiple sites, subjects, and designs
•	Published 15 papers about SimSE in journals, conference proceedings, and workshop proceedings
•	Supervised undergraduate research assistants in developing parts of the SimSE GUI

Google Inc., Santa Monica, CA
Software Engineering Intern 	Summer 2005
Designed and developed automated GUI test scripts for internal applications.


EDUCATION

University of California, Irvine (UCI)	
Doctor of Philosophy in Information and Computer Science	2006
Dissertation: A Software Engineering Simulation Environment for Software Process Education
Master of Science in Information and Computer Science	2003
Bachelor of Science in Biological Sciences	1998


SELECTED PROJECTS 

MoviesStats, AirportsStats (2025): Java applications that make extensive use of Java Streams to analyze a dataset of 23K movies and a dataset of 13K airports and generate various statistics about the data.
Keeper App (2022): Web-based Notes app written in ReactJS, HTML, and CSS
Alien Mastermind (2022): Developed two alien-themed variations of the classic Mastermind game
•	An entirely “front-end” version written in HTML, CSS, and JavaScript using JQuery & Bootstrap.
•	A “full-stack” version with an HTML, CSS, and JavaScript front-end. Java Spring Boot back-end exposes a REST API, uses MongoDB for game persistence. Deployed on Heroku.	
QuizApp (2019): iOS app written in Swift that allows users to develop, manage, and administer quizzes with multiple-choice and numerical fill-in-the blank questions. Supports image-based questions (e.g., captcha identification) w/photo uploads, real-time camera snaps, and in-app drawing. Maintains dynamic scoring, activity logging, and persistence. 2019.


RECOGNITION
•	Featured in UCI Women’s History Month spotlight featuring women in tech, March 2, 2022
•	Lecturer of the Year Honorable Mention (2025, 2024, 2022), Nomination (2019); Excellence in Digital Learning Honorable Mention (2022)
•	Awarded the 2009 Premier Award for Excellence in Education Courseware for SimSE
•	SimSE used in classrooms around the world with over 500 citations in scientific publications


ADDITIONAL INFORMATION

•	Publications: Available online at https://www.ics.uci.edu - publications
•	Personal: Avid reader, runner and stair climber
•	Volunteer: 
o	Judge for Dreams for Schools AppJam, UCI ICS Student Council Jam for Change Hackathon, UCI Anteater Awards
o	Faculty Advisor for 6 student clubs: UCI Management Information Student Society, Blueprint at UCI Club, Developer Student Club, Software Engineering Club at UCI, Superposition UC Irvine Club, Developer Student Club
o	Foster/adoptive family care community leader for non-profit group (OC United)


"""

# 第 2 步：创建 messages 列表（与网站摘要同一形状：system + user）

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
] # fill this in

# 第 3 步：调用 OpenAI（model id 保持 gpt-4.1-mini）
response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages
)

# 第 4 步：打印助手正文
print(response.choices[0].message.content)


## 额外练习：JavaScript 重度站点

你可能发现 `display_summary("https://openai.com")` 效果很差——许多现代站点靠浏览器执行 JS 才渲染正文。可用 **Selenium** / **Playwright** 在真实浏览器里取 HTML，再送进同一套 prompt。社区贡献文件夹里有同学提交的示例。


## 分享你的代码

欢迎把变体放到社区贡献文件夹（Pull Request）。若还不熟 git，可用 GPT 逐步指导提 PR。

专业提示：分享前可在 Jupyter 里 **Edit → Clear All Outputs** 再保存，得到更干净的 diff——但本教学注释任务**禁止**为「清理」而改 `outputs` / `execution_count`。

参考说明：  
https://chatgpt.com/share/677a9cb5-c64c-8012-99e0-e06e88afd293
